# Прогноз восьми горизонтов
Шесть моделей, последовательное обучение из локальных Parquet и финальное смешивание.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, *HERE.parents]
ROOT = next((p for p in CANDIDATES if (p / 'main.ipynb').is_file()
             and (p / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте main.ipynb из папки проекта (рядом с data/raw)')
for name in ('train_solo_track.parquet', 'test_solo_track.parquet'):
    file = ROOT / 'data' / 'raw' / name
    if not file.is_file():
        raise FileNotFoundError(f'Не найден исходный файл: {file}')

for package in ('numpy', 'pandas', 'pyarrow', 'catboost', 'lightgbm',
                'xgboost', 'torch', 'sklearn'):
    if importlib.util.find_spec(package) is None:
        raise ImportError(f'Не установлен {package}; запустите pip install -r requirements.txt в .venv')

SCRIPTS = ROOT / '.cache' / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)
COMPONENTS = ROOT / 'data' / 'processed' / 'components'
COMPONENTS.mkdir(parents=True, exist_ok=True)
(ROOT / 'data' / 'processed' / 'models').mkdir(parents=True, exist_ok=True)

def run_model(filename, expected):
    script = SCRIPTS / filename
    assert script.is_file(), f'Не записан скрипт {script} (выполните предыдущую ячейку)'
    print('Запуск:', filename, '| Python:', sys.executable, flush=True)
    subprocess.run([sys.executable, '-u', str(script)], cwd=str(ROOT), check=True)
    output = COMPONENTS / expected
    assert output.is_file(), f'Модель не создала файл: {output}'
    print('Сохранено:', output, flush=True)

print('Корень проекта:', ROOT)
print('Python:', sys.executable)


# CatBoost88
Один MultiRMSE с 88 признаками. Доля: **50% / 40% / 25%** на горизонтах 1 / 2 / 3–8.

In [ ]:
%%writefile .cache/scripts/cb88.py
from pathlib import Path
import gc, os
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor

STEP = pd.Timedelta(minutes=30)
HORIZON = 8
STATUS_COLS = [f"status_{i}" for i in range(1, 7)]
TARGET = "target_1h"

def validate_data(train, test):
    required_train = {"route_id", "timestamp", TARGET, *STATUS_COLS}
    required_test = {"id", "route_id", "timestamp"}
    if not required_train.issubset(train) or not required_test.issubset(test):
        raise ValueError("Missing required Train/Test columns")
    train = train.copy()
    test = test.copy()  # Keep the original test row order.
    for frame, columns in [(train, required_train), (test, required_test)]:
        if frame[list(columns)].isna().any().any():
            raise ValueError("Missing values in required columns")
        frame["timestamp"] = pd.to_datetime(frame["timestamp"]).astype("datetime64[ns]")
        if frame.duplicated(["route_id", "timestamp"]).any():
            raise ValueError("Duplicate route/timestamp keys")
    if not test.id.is_unique:
        raise ValueError("Duplicate test id")
    values = train[STATUS_COLS + [TARGET]].to_numpy()
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Statuses and target must be finite and nonnegative")
    del values
    train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
    gaps = train.groupby("route_id", sort=False).timestamp.diff().dropna()
    if not gaps.eq(STEP).all():
        raise ValueError("Train must have a complete 30-minute grid per route")
    if set(train.route_id.unique()) != set(test.route_id.unique()):
        raise ValueError("Train/Test route sets differ")
    bounds = train.groupby("route_id", sort=False).timestamp.agg(["min", "max"])
    if bounds["min"].nunique() != 1 or bounds["max"].nunique() != 1:
        raise ValueError("This notebook requires common Train bounds for all routes")
    last = test.route_id.map(bounds["max"])
    steps = (test.timestamp - last) / STEP
    if not steps.isin(range(1, HORIZON + 1)).all():
        raise ValueError("Test timestamps must be steps 1..8 after the last known state")
    if not test.groupby("route_id").size().eq(HORIZON).all():
        raise ValueError("Test must contain exactly eight timestamps per route")
    return train, test


def build_features(train):
    """Input: validated route/time-sorted history. Every feature is causal at its row."""
    group = train.groupby("route_id", sort=False)
    f = {"route_id": pd.Categorical(train.route_id)}
    for col in STATUS_COLS + [TARGET]:
        f[col] = train[col].to_numpy(dtype=np.float32)
    for lag in (1, 2, 4, 8, 48, 96, 336):
        f[f"target_lag_{lag}"] = group[TARGET].shift(lag).to_numpy(dtype=np.float32)
    for window in (4, 8, 48, 336):
        rolling = group[TARGET].rolling(window, min_periods=window)
        f[f"target_mean_{window}"] = rolling.mean().to_numpy(dtype=np.float32)
        f[f"target_std_{window}"] = rolling.std(ddof=0).to_numpy(dtype=np.float32)
    for col in STATUS_COLS:
        for lag in (1, 8, 48):
            f[f"{col}_lag_{lag}"] = group[col].shift(lag).to_numpy(dtype=np.float32)
        for window in (8, 48):
            f[f"{col}_mean_{window}"] = group[col].rolling(window, min_periods=window).mean().to_numpy(dtype=np.float32)
    for lag in (1, 8, 48):
        f[f"target_diff_{lag}"] = f[TARGET] - f[f"target_lag_{lag}"]
    f["target_relative_mean_48"] = f[TARGET] / np.maximum(f["target_mean_48"], 1.0)
    ts = train.timestamp.dt
    f["origin_slot"] = (ts.hour * 2 + ts.minute // 30).to_numpy(dtype=np.float32)
    f["origin_weekday"] = ts.dayofweek.to_numpy(dtype=np.float32)
    f["origin_month"] = ts.month.to_numpy(dtype=np.float32)
    features = pd.DataFrame(f, index=train.index)
    ready = features.notna().all(axis=1).to_numpy()
    return features, ready


def training_indices(train, ready, cutoff, train_days=None, stride=1):
    """Purge complete eight-step label windows; optionally thin origins for backtests."""
    cutoff = pd.Timestamp(cutoff)
    if stride < 1 or int(stride) != stride:
        raise ValueError("stride must be a positive integer")
    latest_origin = cutoff - HORIZON * STEP
    mask = np.asarray(ready, dtype=bool) & train.timestamp.le(latest_origin).to_numpy()
    if train_days is not None:
        mask &= train.timestamp.ge(latest_origin - pd.Timedelta(days=train_days)).to_numpy()
    distance = (latest_origin - train.timestamp) // STEP
    mask &= (distance % stride == 0).to_numpy()
    idx = np.flatnonzero(mask)
    if not len(idx):
        raise ValueError("No eligible training origins; add history or reduce the warmup/window")
    return idx


def future_labels(train, indices, horizons=range(1, 9)):
    group = train.groupby("route_id", sort=False)[TARGET]
    result = np.column_stack([group.shift(-h).iloc[indices].to_numpy(dtype=np.float64) for h in horizons])
    if not np.isfinite(result).all():
        raise ValueError("Some labels are unavailable: check the cutoff and horizon purge")
    return result


def origin_indices(train, ready, cutoff):
    indices = np.flatnonzero(train.timestamp.eq(pd.Timestamp(cutoff)).to_numpy())
    if len(indices) != train.route_id.nunique() or not np.asarray(ready)[indices].all():
        raise ValueError(f"Incomplete route snapshot / feature warmup at {cutoff}")
    return indices


def make_submission(test, forecast):
    if len(forecast) != len(test) or forecast.duplicated(["route_id", "timestamp"]).any():
        raise ValueError("Forecast must contain one prediction for each Test key")
    merged = test[["id", "route_id", "timestamp"]].merge(
        forecast[["route_id", "timestamp", "y_pred"]],
        on=["route_id", "timestamp"], how="left", sort=False, validate="one_to_one")
    pred = merged.y_pred.to_numpy(dtype=np.float64)
    if not merged.id.is_unique or not np.isfinite(pred).all() or (pred < 0).any():
        raise ValueError("Invalid ids, missing predictions, nonfinite or negative values")
    if merged.id.tolist() != test.id.tolist():
        raise ValueError("Submission order changed unexpectedly")
    return merged[["id", "y_pred"]]

def overlap_bounds(y):
    """Forward feasible intervals for y[t] = x[t] + x[t-1] +/- .5.

    Input shape is routes x time. No backwards smoothing or future observations.
    """
    y = np.asarray(y, dtype=np.float64)
    lo, hi = np.zeros_like(y), np.zeros_like(y)
    hi[:, 0] = y[:, 0] + .5
    for t in range(1, y.shape[1]):
        lo[:, t] = np.maximum(0, y[:, t] - .5 - hi[:, t-1])
        hi[:, t] = y[:, t] + .5 - lo[:, t-1]
    return lo, hi


def make_88_features(train):
    base, _ = build_features(train)
    route_counts = train.groupby("route_id", sort=False).size().to_numpy()
    if len(set(route_counts)) != 1 or route_counts[0] < 700:
        raise ValueError("Ожидается полный равномерный временной ряд для каждого маршрута")
    n_routes, n_points = len(route_counts), int(route_counts[0])
    y = train[TARGET].to_numpy(dtype=np.float64).reshape(n_routes, n_points)
    lower, upper = overlap_bounds(y)
    mid = (lower + upper) / 2
    extra = {
        "overlap_current": mid.ravel().astype(np.float32),
        "overlap_lower": lower.ravel().astype(np.float32),
        "overlap_upper": upper.ravel().astype(np.float32),
        "overlap_width": (upper - lower).ravel().astype(np.float32),
    }
    midpoint_group = pd.Series(mid.ravel(), index=train.index).groupby(train.route_id, sort=False)
    for lag in (1, 2, 4, 8, 48, 336):
        extra[f"overlap_lag_{lag}"] = midpoint_group.shift(lag).to_numpy(dtype=np.float32)
    extra["overlap_fraction_current"] = (mid / np.maximum(y, 1)).ravel().astype(np.float32)
    latent_cols = list(extra)
    group = train.groupby("route_id", sort=False)[TARGET]
    weekly_cols = []
    for horizon in range(1, HORIZON + 1):
        name = f"h{horizon}_last_week"
        extra[name] = group.shift(336 - horizon).to_numpy(dtype=np.float32)
        weekly_cols.append(name)
    dynamics = {}
    for window in (4, 8, 48):
        rolling = midpoint_group.rolling(window, min_periods=window)
        dynamics[f"flow_mean_{window}"] = rolling.mean().reset_index(level=0, drop=True).to_numpy(dtype=np.float32)
        dynamics[f"flow_std_{window}"] = rolling.std(ddof=0).reset_index(level=0, drop=True).to_numpy(dtype=np.float32)
    for lag in (1, 4):
        dynamics[f"flow_diff_{lag}"] = mid.ravel().astype(np.float32) - midpoint_group.shift(lag).to_numpy(dtype=np.float32)
    dynamics["flow_relative_mean48"] = mid.ravel().astype(np.float32) / np.maximum(dynamics["flow_mean_48"], 1.)
    frame = pd.concat([base, pd.DataFrame({**extra, **dynamics}, index=train.index)], axis=1)
    selected_cols = list(base.columns) + latent_cols + weekly_cols + list(dynamics)
    assert len(selected_cols) == 88, f"Ожидалось 88 признаков, получено {len(selected_cols)}"
    ready = frame[selected_cols].notna().all(axis=1).to_numpy()
    return frame[selected_cols], ready


def main():
    root = Path.cwd()
    train = pd.read_parquet(root / "data/raw/train_solo_track.parquet")
    test = pd.read_parquet(root / "data/raw/test_solo_track.parquet")
    train, test = validate_data(train, test)
    print("CatBoost88: признаки", flush=True)
    features, ready = make_88_features(train)
    idx = training_indices(train, ready, train.timestamp.max(), train_days=56, stride=1)
    origin = origin_indices(train, ready, train.timestamp.max())
    X_train = features.iloc[idx].copy()
    X_test = features.iloc[origin].copy()
    targets = future_labels(train, idx) / 100_000.0
    assert len(X_train) == len(targets) and X_train.shape[1] == 88
    device = "GPU" if torch.cuda.is_available() else "CPU"
    model = CatBoostRegressor(
        iterations=500, depth=6, learning_rate=0.043006437636033956,
        l2_leaf_reg=2.8432738978688215, border_count=64,
        one_hot_max_size=255, random_seed=42, thread_count=4,
        loss_function="MultiRMSE", task_type=device,
        allow_writing_files=False, use_best_model=False, verbose=False,
    )
    print(f"CatBoost88: {len(idx):,} origins, device={device}", flush=True)
    model.fit(X_train, targets, cat_features=["route_id"])
    model_dir = root / "data/processed/models"
    model_dir.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_dir / "cb88_multi.cbm"))
    pred = np.maximum(model.predict(X_test) * 100_000., 0.)
    cutoff = train.timestamp.max()
    forecasts = pd.DataFrame({
        "route_id": np.repeat(train.iloc[origin].route_id.to_numpy(), HORIZON),
        "timestamp": np.tile(pd.date_range(cutoff + STEP, periods=HORIZON, freq=STEP), len(origin)),
        "y_pred": pred.ravel(),
    })
    result = make_submission(test, forecasts)
    path = root / "data/processed/components/cb88.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    result.sort_values("id").to_csv(path, index=False)
    print("Готово:", path, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
run_model('cb88.py', 'cb88.csv')

# CatBoost A
8 моделей, Huber → MAE. Доля внутри пятикомпонентного ансамбля: **30%**.

In [ ]:
%%writefile .cache/scripts/solution_cb_v25.py
import torch
CB_TASK_TYPE = "GPU" if torch.cuda.is_available() else "CPU"
import os, time, warnings, gc
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET      = "target_1h"
PRED_STEPS  = 8
SEED        = 42
TRAIN_DAYS  = 60

os.makedirs("data/processed/models/cb_v25", exist_ok=True)

def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true > 0
    wape  = np.abs(y_pred[mask] - y_true[mask]).sum() / y_true[mask].sum()
    rbias = abs(y_pred[mask].sum() / y_true[mask].sum() - 1)
    return wape + rbias

# ─── LOAD ──────────────────────────────────────────────────────────────────
print("Loading...")
train = pd.read_parquet("data/raw/train_solo_track.parquet")
test  = pd.read_parquet("data/raw/test_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"]  = pd.to_datetime(test["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test  = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"  train {train.shape}  test {test.shape}")

STATUS_COLS = [c for c in train.columns if c.startswith("status_")]

# ─── FEATURE ENGINEERING ───────────────────────────────────────────────────
print("Feature engineering...")
t1 = time.time()
g = train.groupby("route_id", sort=False)

train["log_target"] = np.log1p(train[TARGET].clip(lower=0))

# Standard lags (same as cb_v12)
for lag in [1, 2, 3, 4, 6, 8, 12, 24, 48, 96, 168, 336, 672]:
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

# NEW: step-adjusted same-time lags (48-s, 336-s, 672-s for s=1..8)
# These let per-step models use "exact same prediction-time anchor" lags
for lag in range(40, 48):   # 40..47 = 48-8 .. 48-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(328, 336): # 328..335 = 336-8 .. 336-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(664, 672): # 664..671 = 672-8 .. 672-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for w in [4, 8, 24, 48, 96, 336]:
    train[f"lt_rm{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=1).mean())
for w in [8, 24, 96]:
    train[f"lt_rs{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=2).std())
for span in [4, 8, 24, 48]:
    train[f"lt_ewm{span}"] = g["log_target"].transform(
        lambda x: x.ewm(span=span, min_periods=1).mean())

train["lt_d1"]   = train["lt_lag1"] - train["lt_lag2"]
train["lt_d4"]   = train["lt_lag1"] - train["lt_lag4"]
train["lt_d48"]  = train["lt_lag1"] - train["lt_lag48"]
train["lt_d336"] = train["lt_lag1"] - train["lt_lag336"]
train["lt_rel8"]  = train["lt_lag1"] - train["lt_rm8"]
train["lt_rel48"] = train["lt_lag1"] - train["lt_rm48"]

for col in STATUS_COLS:
    train[f"{col}_log"] = np.log1p(train[col].clip(lower=0))
    for lag in [1, 2, 4, 8]:
        train[f"{col}_lag{lag}"] = g[col].transform("shift", lag)
    for w in [4, 8]:
        train[f"{col}_rm{w}"] = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())

train["st_local"]    = train[["status_1","status_2","status_3"]].sum(axis=1)
train["st_incoming"] = train[["status_4","status_5","status_6"]].sum(axis=1)
train["st_ratio"]    = np.log1p(train["st_incoming"]) - np.log1p(train["st_local"] + 1)

ts = train["timestamp"]
h  = ts.dt.hour + ts.dt.minute / 60
d  = ts.dt.dayofweek.astype(float)
train["hour_sin"]   = np.sin(2 * np.pi * h / 24)
train["hour_cos"]   = np.cos(2 * np.pi * h / 24)
train["dow_sin"]    = np.sin(2 * np.pi * d / 7)
train["dow_cos"]    = np.cos(2 * np.pi * d / 7)
train["is_weekend"] = (d >= 5).astype(float)

route_stats = g["log_target"].agg(["mean", "std",
                                    lambda x: x.quantile(.1),
                                    lambda x: x.quantile(.9)])
route_stats.columns = ["r_logmean", "r_logstd", "r_logq10", "r_logq90"]
route_stats = route_stats.reset_index()
train = train.merge(route_stats, on="route_id", how="left")

print(f"  Done in {time.time()-t1:.1f}s  cols={len(train.columns)}")

# ─── PER-ROUTE NORMALIZATION ───────────────────────────────────────────────
route_raw_mean = (g[TARGET].mean().clip(lower=1).rename("route_raw_mean").reset_index())
train = train.merge(route_raw_mean, on="route_id", how="left")
train["norm_log_target"] = np.log1p(
    (train[TARGET].clip(lower=0) / train["route_raw_mean"]).clip(lower=0)
)

# ─── FEATURE SET ───────────────────────────────────────────────────────────
exclude = {TARGET, "timestamp", "id", "log_target", "norm_log_target"}
BASE_FEATS = [c for c in train.columns if c not in exclude]
CAT_FEATS  = ["route_id"]
print(f"  Base features: {len(BASE_FEATS)}")

def step_time_feats(origin_ts: pd.Series, step: int) -> pd.DataFrame:
    tgt_ts = origin_ts + pd.Timedelta(minutes=30 * step)
    h = tgt_ts.dt.hour + tgt_ts.dt.minute / 60
    d = tgt_ts.dt.dayofweek.astype(float)
    return pd.DataFrame({
        "tgt_hour_sin": np.sin(2 * np.pi * h / 24),
        "tgt_hour_cos": np.cos(2 * np.pi * h / 24),
        "tgt_dow_sin":  np.sin(2 * np.pi * d / 7),
        "tgt_dow_cos":  np.cos(2 * np.pi * d / 7),
        "tgt_is_wknd":  (d >= 5).astype(float),
    }, index=origin_ts.index)

# ─── VALIDATION SPLIT ──────────────────────────────────────────────────────
ts_max      = train["timestamp"].max()
CUTOFF      = ts_max - pd.Timedelta(days=7)
TRAIN_START = CUTOFF - pd.Timedelta(days=TRAIN_DAYS)
train_mask  = (train["timestamp"] > TRAIN_START) & (train["timestamp"] <= CUTOFF)
val_mask    = train["timestamp"] >  CUTOFF
full_mask   = train["timestamp"] >  TRAIN_START
g_full      = train.groupby("route_id", sort=False)
print(f"  Training window: {TRAIN_START.date()} → {CUTOFF.date()} ({TRAIN_DAYS}d)")

# ─── PREPARE TEST BASE ─────────────────────────────────────────────────────
last_rows_df = (train.groupby("route_id", sort=False)
                     .tail(1).reset_index(drop=True).copy())
test_routes  = test["route_id"].values
test_X_base  = (pd.DataFrame({"route_id": test_routes})
                .merge(last_rows_df[["route_id"] + [f for f in BASE_FEATS if f != "route_id"]],
                       on="route_id", how="left")[BASE_FEATS]
                .reset_index(drop=True))

test_ts   = test["timestamp"]
step_col  = ((test_ts - train["timestamp"].max()) / pd.Timedelta("30min")).astype(int)
test_preds = np.zeros(len(test))

val_preds_by_route_step = {}
val_true_by_route_step  = {}

FEAT_COLS_STEP = BASE_FEATS + ["tgt_hour_sin","tgt_hour_cos","tgt_dow_sin","tgt_dow_cos","tgt_is_wknd"]

# ─── TRAIN 8 PER-STEP MODELS ──────────────────────────────────────────────
print("\n=== Training 8 per-step CatBoost models (+ step-adjusted lags) ===\n")

for step in range(1, PRED_STEPS + 1):
    print(f"\n--- Step {step} / {PRED_STEPS}  (lags: same-day={48-step}, same-week={336-step}, 2wk={672-step}) ---")
    t_step = time.time()

    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)

    valid   = y_log.notna()
    mask_tr = train_mask & valid
    mask_va = val_mask   & valid
    mask_fl = full_mask  & valid

    X_tr_s = X_step.loc[mask_tr].copy()
    y_tr_s = y_log.loc[mask_tr]
    X_va_s = X_step.loc[mask_va].copy()
    y_va_s = y_log.loc[mask_va]
    for c in CAT_FEATS:
        X_tr_s[c] = X_tr_s[c].astype(str)
        X_va_s[c] = X_va_s[c].astype(str)

    model_val = CatBoostRegressor(
        iterations            = 3000,
        depth                 = 8,
        loss_function         = "Huber:delta=1.0",
        eval_metric           = "Huber:delta=1.0",
        random_seed           = SEED,
        task_type             = CB_TASK_TYPE,
        early_stopping_rounds = 150,
        use_best_model        = True,
        verbose               = 0,
    )
    model_val.fit(X_tr_s, y_tr_s, eval_set=(X_va_s, y_va_s), cat_features=CAT_FEATS)
    best_n = max(1, int(model_val.best_iteration_))

    pred_va = np.expm1(model_val.predict(X_va_s)).clip(0)
    val_s   = wape_rbias(np.expm1(y_va_s.values), pred_va)
    print(f"  best_n={best_n}  val={val_s:.4f}")

    del model_val, X_tr_s, y_tr_s, X_va_s
    gc.collect()

    X_fl_s = X_step.loc[mask_fl].copy()
    y_fl_s = y_log.loc[mask_fl]
    for c in CAT_FEATS:
        X_fl_s[c] = X_fl_s[c].astype(str)

    model_final = CatBoostRegressor(
        iterations    = best_n,
        depth         = 8,
        loss_function = "MAE",
        random_seed   = SEED,
        task_type     = CB_TASK_TYPE,
        verbose       = 0,
    )
    model_final.fit(X_fl_s, y_fl_s, cat_features=CAT_FEATS)

    del X_fl_s, y_fl_s
    gc.collect()

    model_path = f"data/processed/models/cb_v25/step{step}.cbm"
    model_final.save_model(model_path)
    print(f"  Saved → {model_path}")

    st_test = step_time_feats(test_ts, step)
    X_test  = pd.concat([test_X_base.reset_index(drop=True),
                         st_test.reset_index(drop=True)], axis=1)[FEAT_COLS_STEP].copy()
    X_test["route_id"] = X_test["route_id"].astype(str)

    pred_norm   = np.expm1(model_final.predict(X_test)).clip(0)
    route_means = (pd.DataFrame({"route_id": test_routes})
                   .merge(route_raw_mean, on="route_id", how="left")["route_raw_mean"].values)
    step_mask_idx = (step_col == step).values
    test_preds[step_mask_idx] = (pred_norm * route_means)[step_mask_idx]

    X_va2 = X_step.loc[mask_va].copy()
    X_va2["route_id"] = X_va2["route_id"].astype(str)
    p_norm_v  = np.expm1(model_final.predict(X_va2[FEAT_COLS_STEP])).clip(0)
    r_means_v = train.loc[mask_va, "route_raw_mean"].values
    routes_v  = train.loc[mask_va, "route_id"].values
    p_raw = p_norm_v * r_means_v
    t_raw = np.expm1(y_log.loc[mask_va].values) * r_means_v
    for r, p, t in zip(routes_v, p_raw, t_raw):
        key = (r, step)
        val_preds_by_route_step.setdefault(key, []).append(p)
        val_true_by_route_step.setdefault(key,  []).append(t)

    del model_final, X_va2
    gc.collect()
    print(f"  Step {step} done in {time.time()-t_step:.1f}s")

# ─── PER-ROUTE × PER-STEP BIAS CORRECTION ─────────────────────────────────
print("\nApplying per-route × per-step bias correction...")
scale      = {}
step_scale = {}
for step in range(1, PRED_STEPS + 1):
    all_p, all_t = [], []
    for route in np.unique(test_routes):
        key   = (route, step)
        p_arr = np.array(val_preds_by_route_step.get(key, []))
        t_arr = np.array(val_true_by_route_step.get(key,  []))
        if len(p_arr) > 0 and p_arr.sum() > 0:
            scale[key] = np.clip(t_arr.sum() / p_arr.sum(), 0.7, 1.4)
        all_p.extend(val_preds_by_route_step.get(key, []))
        all_t.extend(val_true_by_route_step.get(key,  []))
    step_scale[step] = np.clip(sum(all_t) / max(sum(all_p), 1e-6), 0.7, 1.4)

for step in range(1, PRED_STEPS + 1):
    idx_here    = np.where((step_col == step).values)[0]
    routes_here = test_routes[idx_here]
    for idx, route in zip(idx_here, routes_here):
        test_preds[idx] *= scale.get((route, step), step_scale[step])

all_p_v = [v for vals in val_preds_by_route_step.values() for v in vals]
all_t_v = [v for vals in val_true_by_route_step.values()  for v in vals]
print(f"  Val WAPE+RBias (before bias corr): {wape_rbias(np.array(all_t_v), np.array(all_p_v)):.4f}")

test_preds = np.clip(test_preds, 0, None)

# ─── SUBMISSION ────────────────────────────────────────────────────────────
sub = test[["id"]].copy()
sub["y_pred"] = test_preds.round(0)
sub = sub.sort_values("id").reset_index(drop=True)

assert sub["id"].isna().sum() == 0
assert sub["y_pred"].isna().sum() == 0
assert (sub["y_pred"] < 0).sum() == 0

out = "data/processed/components/submission_cb_v25.csv"
sub.to_csv(out, index=False)
print(f"\n{'='*60}")
print(f"Saved → {out}  shape={sub.shape}")
print(f"\nStats:\n{sub['y_pred'].describe().round(0)}")
print(f"\nTotal: {(time.time()-T0)/60:.1f} min")


In [ ]:
run_model('solution_cb_v25.py', 'submission_cb_v25.csv')

# CatBoost B
8 моделей, Tweedie. Доля внутри пятикомпонентного ансамбля: **30%**.

In [ ]:
%%writefile .cache/scripts/solution_cb_v36.py
import torch
CB_TASK_TYPE = "GPU" if torch.cuda.is_available() else "CPU"
import os, time, warnings, gc
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET      = "target_1h"
PRED_STEPS  = 8
SEED        = 42
TRAIN_DAYS  = 60

os.makedirs("data/processed/models/cb_v36", exist_ok=True)

def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true > 0
    wape  = np.abs(y_pred[mask] - y_true[mask]).sum() / y_true[mask].sum()
    rbias = abs(y_pred[mask].sum() / y_true[mask].sum() - 1)
    return wape + rbias

# ─── LOAD ──────────────────────────────────────────────────────────────────
print("Loading...")
train = pd.read_parquet("data/raw/train_solo_track.parquet")
test  = pd.read_parquet("data/raw/test_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"]  = pd.to_datetime(test["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test  = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"  train {train.shape}  test {test.shape}")

STATUS_COLS = [c for c in train.columns if c.startswith("status_")]

# ─── FEATURE ENGINEERING ───────────────────────────────────────────────────
print("Feature engineering...")
t1 = time.time()
g = train.groupby("route_id", sort=False)

train["log_target"] = np.log1p(train[TARGET].clip(lower=0))

# Standard lags (same as cb_v12)
for lag in [1, 2, 3, 4, 6, 8, 12, 24, 48, 96, 168, 336, 672]:
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

# NEW: step-adjusted same-time lags (48-s, 336-s, 672-s for s=1..8)
# These let per-step models use "exact same prediction-time anchor" lags
for lag in range(40, 48):   # 40..47 = 48-8 .. 48-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(328, 336): # 328..335 = 336-8 .. 336-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(664, 672): # 664..671 = 672-8 .. 672-1
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for w in [4, 8, 24, 48, 96, 336]:
    train[f"lt_rm{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=1).mean())
for w in [8, 24, 96]:
    train[f"lt_rs{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=2).std())
for span in [4, 8, 24, 48]:
    train[f"lt_ewm{span}"] = g["log_target"].transform(
        lambda x: x.ewm(span=span, min_periods=1).mean())

train["lt_d1"]   = train["lt_lag1"] - train["lt_lag2"]
train["lt_d4"]   = train["lt_lag1"] - train["lt_lag4"]
train["lt_d48"]  = train["lt_lag1"] - train["lt_lag48"]
train["lt_d336"] = train["lt_lag1"] - train["lt_lag336"]
train["lt_rel8"]  = train["lt_lag1"] - train["lt_rm8"]
train["lt_rel48"] = train["lt_lag1"] - train["lt_rm48"]

for col in STATUS_COLS:
    train[f"{col}_log"] = np.log1p(train[col].clip(lower=0))
    for lag in [1, 2, 4, 8]:
        train[f"{col}_lag{lag}"] = g[col].transform("shift", lag)
    for w in [4, 8]:
        train[f"{col}_rm{w}"] = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())

train["st_local"]    = train[["status_1","status_2","status_3"]].sum(axis=1)
train["st_incoming"] = train[["status_4","status_5","status_6"]].sum(axis=1)
train["st_ratio"]    = np.log1p(train["st_incoming"]) - np.log1p(train["st_local"] + 1)

ts = train["timestamp"]
h  = ts.dt.hour + ts.dt.minute / 60
d  = ts.dt.dayofweek.astype(float)
train["hour_sin"]   = np.sin(2 * np.pi * h / 24)
train["hour_cos"]   = np.cos(2 * np.pi * h / 24)
train["dow_sin"]    = np.sin(2 * np.pi * d / 7)
train["dow_cos"]    = np.cos(2 * np.pi * d / 7)
train["is_weekend"] = (d >= 5).astype(float)

route_stats = g["log_target"].agg(["mean", "std",
                                    lambda x: x.quantile(.1),
                                    lambda x: x.quantile(.9)])
route_stats.columns = ["r_logmean", "r_logstd", "r_logq10", "r_logq90"]
route_stats = route_stats.reset_index()
train = train.merge(route_stats, on="route_id", how="left")

print(f"  Done in {time.time()-t1:.1f}s  cols={len(train.columns)}")

# ─── PER-ROUTE NORMALIZATION ───────────────────────────────────────────────
route_raw_mean = (g[TARGET].mean().clip(lower=1).rename("route_raw_mean").reset_index())
train = train.merge(route_raw_mean, on="route_id", how="left")
train["norm_log_target"] = np.log1p(
    (train[TARGET].clip(lower=0) / train["route_raw_mean"]).clip(lower=0)
)

# ─── FEATURE SET ───────────────────────────────────────────────────────────
exclude = {TARGET, "timestamp", "id", "log_target", "norm_log_target"}
BASE_FEATS = [c for c in train.columns if c not in exclude]
CAT_FEATS  = ["route_id"]
print(f"  Base features: {len(BASE_FEATS)}")

def step_time_feats(origin_ts: pd.Series, step: int) -> pd.DataFrame:
    tgt_ts = origin_ts + pd.Timedelta(minutes=30 * step)
    h = tgt_ts.dt.hour + tgt_ts.dt.minute / 60
    d = tgt_ts.dt.dayofweek.astype(float)
    return pd.DataFrame({
        "tgt_hour_sin": np.sin(2 * np.pi * h / 24),
        "tgt_hour_cos": np.cos(2 * np.pi * h / 24),
        "tgt_dow_sin":  np.sin(2 * np.pi * d / 7),
        "tgt_dow_cos":  np.cos(2 * np.pi * d / 7),
        "tgt_is_wknd":  (d >= 5).astype(float),
    }, index=origin_ts.index)

# ─── VALIDATION SPLIT ──────────────────────────────────────────────────────
ts_max      = train["timestamp"].max()
CUTOFF      = ts_max - pd.Timedelta(days=7)
TRAIN_START = CUTOFF - pd.Timedelta(days=TRAIN_DAYS)
train_mask  = (train["timestamp"] > TRAIN_START) & (train["timestamp"] <= CUTOFF)
val_mask    = train["timestamp"] >  CUTOFF
full_mask   = train["timestamp"] >  TRAIN_START
g_full      = train.groupby("route_id", sort=False)
print(f"  Training window: {TRAIN_START.date()} → {CUTOFF.date()} ({TRAIN_DAYS}d)")

# ─── PREPARE TEST BASE ─────────────────────────────────────────────────────
last_rows_df = (train.groupby("route_id", sort=False)
                     .tail(1).reset_index(drop=True).copy())
test_routes  = test["route_id"].values
test_X_base  = (pd.DataFrame({"route_id": test_routes})
                .merge(last_rows_df[["route_id"] + [f for f in BASE_FEATS if f != "route_id"]],
                       on="route_id", how="left")[BASE_FEATS]
                .reset_index(drop=True))

test_ts   = test["timestamp"]
step_col  = ((test_ts - train["timestamp"].max()) / pd.Timedelta("30min")).astype(int)
test_preds = np.zeros(len(test))

val_preds_by_route_step = {}
val_true_by_route_step  = {}

FEAT_COLS_STEP = BASE_FEATS + ["tgt_hour_sin","tgt_hour_cos","tgt_dow_sin","tgt_dow_cos","tgt_is_wknd"]

# ─── TRAIN 8 PER-STEP MODELS ──────────────────────────────────────────────
print("\n=== Training 8 per-step CatBoost models (+ step-adjusted lags) ===\n")

for step in range(1, PRED_STEPS + 1):
    print(f"\n--- Step {step} / {PRED_STEPS}  (lags: same-day={48-step}, same-week={336-step}, 2wk={672-step}) ---")
    t_step = time.time()

    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)

    valid   = y_log.notna()
    mask_tr = train_mask & valid
    mask_va = val_mask   & valid
    mask_fl = full_mask  & valid

    X_tr_s = X_step.loc[mask_tr].copy()
    y_tr_s = y_log.loc[mask_tr]
    X_va_s = X_step.loc[mask_va].copy()
    y_va_s = y_log.loc[mask_va]
    for c in CAT_FEATS:
        X_tr_s[c] = X_tr_s[c].astype(str)
        X_va_s[c] = X_va_s[c].astype(str)

    model_val = CatBoostRegressor(
        iterations            = 3000,
        depth                 = 8,
        loss_function         = "Tweedie:variance_power=1.5",
        eval_metric           = "Tweedie:variance_power=1.5",
        random_seed           = SEED,
        task_type             = CB_TASK_TYPE,
        early_stopping_rounds = 150,
        use_best_model        = True,
        verbose               = 0,
    )
    model_val.fit(X_tr_s, y_tr_s, eval_set=(X_va_s, y_va_s), cat_features=CAT_FEATS)
    best_n = max(1, int(model_val.best_iteration_))

    pred_va = np.expm1(model_val.predict(X_va_s)).clip(0)
    val_s   = wape_rbias(np.expm1(y_va_s.values), pred_va)
    print(f"  best_n={best_n}  val={val_s:.4f}")

    del model_val, X_tr_s, y_tr_s, X_va_s
    gc.collect()

    X_fl_s = X_step.loc[mask_fl].copy()
    y_fl_s = y_log.loc[mask_fl]
    for c in CAT_FEATS:
        X_fl_s[c] = X_fl_s[c].astype(str)

    model_final = CatBoostRegressor(
        iterations    = best_n,
        depth         = 8,
        loss_function = "Tweedie:variance_power=1.5",
        random_seed   = SEED,
        task_type     = CB_TASK_TYPE,
        verbose       = 0,
    )
    model_final.fit(X_fl_s, y_fl_s, cat_features=CAT_FEATS)

    del X_fl_s, y_fl_s
    gc.collect()

    model_path = f"data/processed/models/cb_v36/step{step}.cbm"
    model_final.save_model(model_path)
    print(f"  Saved → {model_path}")

    st_test = step_time_feats(test_ts, step)
    X_test  = pd.concat([test_X_base.reset_index(drop=True),
                         st_test.reset_index(drop=True)], axis=1)[FEAT_COLS_STEP].copy()
    X_test["route_id"] = X_test["route_id"].astype(str)

    pred_norm   = np.expm1(model_final.predict(X_test)).clip(0)
    route_means = (pd.DataFrame({"route_id": test_routes})
                   .merge(route_raw_mean, on="route_id", how="left")["route_raw_mean"].values)
    step_mask_idx = (step_col == step).values
    test_preds[step_mask_idx] = (pred_norm * route_means)[step_mask_idx]

    X_va2 = X_step.loc[mask_va].copy()
    X_va2["route_id"] = X_va2["route_id"].astype(str)
    p_norm_v  = np.expm1(model_final.predict(X_va2[FEAT_COLS_STEP])).clip(0)
    r_means_v = train.loc[mask_va, "route_raw_mean"].values
    routes_v  = train.loc[mask_va, "route_id"].values
    p_raw = p_norm_v * r_means_v
    t_raw = np.expm1(y_log.loc[mask_va].values) * r_means_v
    for r, p, t in zip(routes_v, p_raw, t_raw):
        key = (r, step)
        val_preds_by_route_step.setdefault(key, []).append(p)
        val_true_by_route_step.setdefault(key,  []).append(t)

    del model_final, X_va2
    gc.collect()
    print(f"  Step {step} done in {time.time()-t_step:.1f}s")

# ─── PER-ROUTE × PER-STEP BIAS CORRECTION ─────────────────────────────────
print("\nApplying per-route × per-step bias correction...")
scale      = {}
step_scale = {}
for step in range(1, PRED_STEPS + 1):
    all_p, all_t = [], []
    for route in np.unique(test_routes):
        key   = (route, step)
        p_arr = np.array(val_preds_by_route_step.get(key, []))
        t_arr = np.array(val_true_by_route_step.get(key,  []))
        if len(p_arr) > 0 and p_arr.sum() > 0:
            scale[key] = np.clip(t_arr.sum() / p_arr.sum(), 0.7, 1.4)
        all_p.extend(val_preds_by_route_step.get(key, []))
        all_t.extend(val_true_by_route_step.get(key,  []))
    step_scale[step] = np.clip(sum(all_t) / max(sum(all_p), 1e-6), 0.7, 1.4)

for step in range(1, PRED_STEPS + 1):
    idx_here    = np.where((step_col == step).values)[0]
    routes_here = test_routes[idx_here]
    for idx, route in zip(idx_here, routes_here):
        test_preds[idx] *= scale.get((route, step), step_scale[step])

all_p_v = [v for vals in val_preds_by_route_step.values() for v in vals]
all_t_v = [v for vals in val_true_by_route_step.values()  for v in vals]
print(f"  Val WAPE+RBias (before bias corr): {wape_rbias(np.array(all_t_v), np.array(all_p_v)):.4f}")

test_preds = np.clip(test_preds, 0, None)

# ─── SUBMISSION ────────────────────────────────────────────────────────────
sub = test[["id"]].copy()
sub["y_pred"] = test_preds.round(0)
sub = sub.sort_values("id").reset_index(drop=True)

assert sub["id"].isna().sum() == 0
assert sub["y_pred"].isna().sum() == 0
assert (sub["y_pred"] < 0).sum() == 0

out = "data/processed/components/submission_cb_v36.csv"
sub.to_csv(out, index=False)
print(f"\n{'='*60}")
print(f"Saved → {out}  shape={sub.shape}")
print(f"\nStats:\n{sub['y_pred'].describe().round(0)}")
print(f"\nTotal: {(time.time()-T0)/60:.1f} min")


In [ ]:
run_model('solution_cb_v36.py', 'submission_cb_v36.csv')

# LightGBM
Общая модель с шагом прогноза, Huber → MAE. Доля внутри ансамбля: **10%**.

In [ ]:
%%writefile .cache/scripts/solution_lgb_v1.py
import os, time, warnings, gc
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET      = "target_1h"
PRED_STEPS  = 8
SEED        = 42
TRAIN_DAYS  = 60

os.makedirs("data/processed/models/lgb_v1", exist_ok=True)

# ─── METRIC ────────────────────────────────────────────────────────────────
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true > 0
    wape  = np.abs(y_pred[mask] - y_true[mask]).sum() / y_true[mask].sum()
    rbias = abs(y_pred[mask].sum() / y_true[mask].sum() - 1)
    return wape + rbias

# ─── LOAD ──────────────────────────────────────────────────────────────────
print("Loading...")
train = pd.read_parquet("data/raw/train_solo_track.parquet")
test  = pd.read_parquet("data/raw/test_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"]  = pd.to_datetime(test["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test  = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"  train {train.shape}  test {test.shape}")

STATUS_COLS = [c for c in train.columns if c.startswith("status_")]

# ─── ROUTE ID LABEL ENCODING ────────────────────────────────────────────────
le = LabelEncoder()
all_routes = pd.concat([train["route_id"], test["route_id"]]).unique()
le.fit(all_routes)
train["route_id_enc"] = le.transform(train["route_id"])
test["route_id_enc"]  = le.transform(test["route_id"])

# ─── FEATURE ENGINEERING ───────────────────────────────────────────────────
print("Feature engineering...")
t1 = time.time()
g = train.groupby("route_id", sort=False)

train["log_target"] = np.log1p(train[TARGET].clip(lower=0))

for lag in [1, 2, 3, 4, 6, 8, 12, 24, 48, 96, 168, 336, 672]:
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for w in [4, 8, 24, 48, 96, 336]:
    train[f"lt_rm{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=1).mean())
for w in [8, 24, 96]:
    train[f"lt_rs{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=2).std())

for span in [4, 8, 24, 48]:
    train[f"lt_ewm{span}"] = g["log_target"].transform(
        lambda x: x.ewm(span=span, min_periods=1).mean())

train["lt_d1"]   = train["lt_lag1"] - train["lt_lag2"]
train["lt_d4"]   = train["lt_lag1"] - train["lt_lag4"]
train["lt_d48"]  = train["lt_lag1"] - train["lt_lag48"]
train["lt_d336"] = train["lt_lag1"] - train["lt_lag336"]
train["lt_rel8"]  = train["lt_lag1"] - train["lt_rm8"]
train["lt_rel48"] = train["lt_lag1"] - train["lt_rm48"]

for col in STATUS_COLS:
    train[f"{col}_log"] = np.log1p(train[col].clip(lower=0))
    for lag in [1, 2, 4, 8]:
        train[f"{col}_lag{lag}"] = g[col].transform("shift", lag)
    for w in [4, 8]:
        train[f"{col}_rm{w}"] = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())

train["st_local"]    = train[["status_1","status_2","status_3"]].sum(axis=1)
train["st_incoming"] = train[["status_4","status_5","status_6"]].sum(axis=1)
train["st_ratio"]    = np.log1p(train["st_incoming"]) - np.log1p(train["st_local"] + 1)

ts = train["timestamp"]
h  = ts.dt.hour + ts.dt.minute / 60
d  = ts.dt.dayofweek.astype(float)
train["hour_sin"]   = np.sin(2 * np.pi * h / 24)
train["hour_cos"]   = np.cos(2 * np.pi * h / 24)
train["dow_sin"]    = np.sin(2 * np.pi * d / 7)
train["dow_cos"]    = np.cos(2 * np.pi * d / 7)
train["is_weekend"] = (d >= 5).astype(float)

route_stats = g["log_target"].agg(["mean", "std",
                                    lambda x: x.quantile(.1),
                                    lambda x: x.quantile(.9)])
route_stats.columns = ["r_logmean", "r_logstd", "r_logq10", "r_logq90"]
route_stats = route_stats.reset_index()
train = train.merge(route_stats, on="route_id", how="left")

print(f"  Done in {time.time()-t1:.1f}s  cols={len(train.columns)}")

# ─── PER-ROUTE NORMALIZATION ───────────────────────────────────────────────
route_raw_mean = (g[TARGET].mean().clip(lower=1).rename("route_raw_mean").reset_index())
train = train.merge(route_raw_mean, on="route_id", how="left")
train["norm_log_target"] = np.log1p(
    (train[TARGET].clip(lower=0) / train["route_raw_mean"]).clip(lower=0)
)

# ─── FEATURE SET ───────────────────────────────────────────────────────────
# Use route_id_enc (int) instead of route_id string — GPU requires ≤255 bins
exclude = {TARGET, "timestamp", "id", "log_target", "norm_log_target", "route_id"}
BASE_FEATS = [c for c in train.columns if c not in exclude]
print(f"  Features: {len(BASE_FEATS)}")

# ─── STEP-TIME FEATURES ────────────────────────────────────────────────────
def step_time_feats(origin_ts: pd.Series, step: int) -> pd.DataFrame:
    tgt_ts = origin_ts + pd.Timedelta(minutes=30 * step)
    h = tgt_ts.dt.hour + tgt_ts.dt.minute / 60
    d = tgt_ts.dt.dayofweek.astype(float)
    return pd.DataFrame({
        "tgt_hour_sin": np.sin(2 * np.pi * h / 24),
        "tgt_hour_cos": np.cos(2 * np.pi * h / 24),
        "tgt_dow_sin":  np.sin(2 * np.pi * d / 7),
        "tgt_dow_cos":  np.cos(2 * np.pi * d / 7),
        "tgt_is_wknd":  (d >= 5).astype(float),
        "step":         float(step),
    }, index=origin_ts.index)

# ─── VALIDATION SPLIT ──────────────────────────────────────────────────────
ts_max       = train["timestamp"].max()
CUTOFF       = ts_max - pd.Timedelta(days=7)
TRAIN_START  = CUTOFF - pd.Timedelta(days=TRAIN_DAYS)
train_mask   = (train["timestamp"] > TRAIN_START) & (train["timestamp"] <= CUTOFF)
val_mask     = train["timestamp"] >  CUTOFF
g_full       = train.groupby("route_id", sort=False)
print(f"  Training window: {TRAIN_START.date()} → {CUTOFF.date()} ({TRAIN_DAYS}d)")

# ─── BUILD TRAINING DATA ────────────────────────────────────────────────────
print("\nBuilding combined multi-step dataset...")
t2 = time.time()

all_X_tr, all_y_tr = [], []
all_X_va, all_y_va = [], []

for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)

    valid   = y_log.notna()
    mask_tr = train_mask & valid
    mask_va = val_mask   & valid

    all_X_tr.append(X_step.loc[mask_tr])
    all_y_tr.append(y_log.loc[mask_tr])
    all_X_va.append(X_step.loc[mask_va])
    all_y_va.append(y_log.loc[mask_va])

X_tr = pd.concat(all_X_tr, ignore_index=True)
y_tr = pd.concat(all_y_tr, ignore_index=True)
X_va = pd.concat(all_X_va, ignore_index=True)
y_va = pd.concat(all_y_va, ignore_index=True)

print(f"  train={len(X_tr):,}  val={len(X_va):,}  done in {time.time()-t2:.1f}s")

del all_X_tr, all_X_va, all_y_tr, all_y_va
gc.collect()

FEAT_COLS = BASE_FEATS + ["tgt_hour_sin", "tgt_hour_cos", "tgt_dow_sin",
                           "tgt_dow_cos", "tgt_is_wknd", "step"]

# ─── TRAIN (val model for early stopping) ──────────────────────────────────
print("\n=== Training LightGBM (Huber, GPU) ===")

model_val = lgb.LGBMRegressor(
    n_estimators      = 3000,
    max_depth         = 8,
    num_leaves        = 255,
    max_bin           = 255,
    objective         = "huber",
    alpha             = 1.0,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    device_type       = "cpu",
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)
model_val.fit(
    X_tr[FEAT_COLS], y_tr,
    eval_set  = [(X_va[FEAT_COLS], y_va)],
    callbacks = [lgb.early_stopping(150, verbose=True), lgb.log_evaluation(200)],
)
best_n = max(1, int(model_val.best_iteration_))
print(f"\n  best_iteration = {best_n}")

pred_va_log = model_val.predict(X_va[FEAT_COLS])
pred_va_raw = np.expm1(pred_va_log).clip(0)
y_va_raw    = np.expm1(y_va.values)
val_score   = wape_rbias(y_va_raw, pred_va_raw)
print(f"  Val WAPE+RBias = {val_score:.4f}")

# ─── RETRAIN (MAE, best_n iters) ───────────────────────────────────────────
print(f"\nRetraining on last {TRAIN_DAYS}d + val (iterations={best_n})...")
t3 = time.time()

full_mask = train["timestamp"] > TRAIN_START

all_X_full, all_y_full = [], []
for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)
    valid  = y_log.notna() & full_mask
    all_X_full.append(X_step.loc[valid])
    all_y_full.append(y_log.loc[valid])

X_full = pd.concat(all_X_full, ignore_index=True)
y_full = pd.concat(all_y_full, ignore_index=True)

del all_X_full, all_y_full
gc.collect()

model_final = lgb.LGBMRegressor(
    n_estimators      = best_n,
    max_depth         = 8,
    num_leaves        = 255,
    max_bin           = 255,
    objective         = "regression_l1",
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    device_type       = "cpu",
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)
model_final.fit(X_full[FEAT_COLS], y_full)
print(f"  Done in {time.time()-t3:.1f}s")

model_path = "data/processed/models/lgb_v1/lgb_v1_final.txt"
model_final.booster_.save_model(model_path)
print(f"  Saved → {model_path}")

del X_full, y_full, X_tr, y_tr, X_va, y_va
gc.collect()

# ─── PREDICT TEST ──────────────────────────────────────────────────────────
print("\n=== Predicting test ===")

last_rows_df = (train.groupby("route_id", sort=False)
                     .tail(1).reset_index(drop=True).copy())

test_routes     = test["route_id"].values
test_routes_enc = test["route_id_enc"].values

_merge_cols = ["route_id"] + [f for f in BASE_FEATS if f != "route_id_enc"]
test_X_base = (pd.DataFrame({"route_id": test_routes})
               .merge(last_rows_df[["route_id"] + [f for f in BASE_FEATS]], on="route_id", how="left")
               [BASE_FEATS].reset_index(drop=True))
test_X_base["route_id_enc"] = test_routes_enc

test_ts  = test["timestamp"]
step_col = ((test_ts - train["timestamp"].max()) / pd.Timedelta("30min")).astype(int)

test_preds = np.zeros(len(test))

for step in range(1, PRED_STEPS + 1):
    step_mask  = (step_col == step).values
    st_df_test = step_time_feats(test_ts, step)
    X_test = pd.concat([
        test_X_base.reset_index(drop=True),
        st_df_test.reset_index(drop=True)
    ], axis=1)[FEAT_COLS].copy()

    pred_norm = np.expm1(model_final.predict(X_test)).clip(0)
    route_means = (pd.DataFrame({"route_id": test_routes})
                   .merge(route_raw_mean, on="route_id", how="left")["route_raw_mean"].values)
    pred_raw = pred_norm * route_means
    test_preds[step_mask] = pred_raw[step_mask]

# ─── BIAS CORRECTION ──────────────────────────────────────────────────────
val_all_pred, val_all_true = [], []
for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)
    valid  = y_log.notna() & val_mask
    X_v    = X_step.loc[valid][FEAT_COLS].copy()
    p_norm = np.expm1(model_final.predict(X_v)).clip(0)
    r_means_v = train.loc[valid, "route_raw_mean"].values
    val_all_pred.append(p_norm * r_means_v)
    val_all_true.append(np.expm1(y_log.loc[valid].values) * r_means_v)

y_v_all = np.concatenate(val_all_true)
p_v_all = np.concatenate(val_all_pred)
final_val = wape_rbias(y_v_all, p_v_all)
bias      = y_v_all.sum() / max(p_v_all.sum(), 1)
print(f"  Val WAPE+RBias (final model): {final_val:.4f}")
print(f"  Bias factor: {bias:.4f}")
if abs(bias - 1) > 0.01:
    test_preds *= bias
    print(f"  Bias correction applied.")

test_preds = np.clip(test_preds, 0, None)

# ─── SUBMISSION ────────────────────────────────────────────────────────────
sub = test[["id"]].copy()
sub["y_pred"] = test_preds.round(0)
sub = sub.sort_values("id").reset_index(drop=True)

assert sub["id"].isna().sum() == 0
assert sub["y_pred"].isna().sum() == 0
assert (sub["y_pred"] < 0).sum() == 0

out = "data/processed/components/submission_lgb_v1.csv"
sub.to_csv(out, index=False)
print(f"\n{'='*60}")
print(f"Saved → {out}  shape={sub.shape}")
print(sub.head(16).to_string())
print(f"\nStats:\n{sub['y_pred'].describe().round(0)}")
print(f"\nTotal: {(time.time()-T0)/60:.1f} min")


In [ ]:
run_model('solution_lgb_v1.py', 'submission_lgb_v1.csv')

# XGBoost
Общая модель с шагом прогноза, Pseudo-Huber → MAE. Доля внутри ансамбля: **10%**.

In [ ]:
%%writefile .cache/scripts/solution_xgb_v1.py
import torch
XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import os, time, warnings, gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET      = "target_1h"
PRED_STEPS  = 8
SEED        = 42
TRAIN_DAYS  = 60

os.makedirs("data/processed/models/xgb_v1", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

# ─── METRIC ────────────────────────────────────────────────────────────────
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true > 0
    wape  = np.abs(y_pred[mask] - y_true[mask]).sum() / y_true[mask].sum()
    rbias = abs(y_pred[mask].sum() / y_true[mask].sum() - 1)
    return wape + rbias

# ─── LOAD ──────────────────────────────────────────────────────────────────
print("Loading...")
train = pd.read_parquet("data/raw/train_solo_track.parquet")
test  = pd.read_parquet("data/raw/test_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"]  = pd.to_datetime(test["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test  = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"  train {train.shape}  test {test.shape}")

STATUS_COLS = [c for c in train.columns if c.startswith("status_")]

# ─── ROUTE ID LABEL ENCODING ────────────────────────────────────────────────
le = LabelEncoder()
all_routes = pd.concat([train["route_id"], test["route_id"]]).unique()
le.fit(all_routes)
train["route_id_enc"] = le.transform(train["route_id"])
test["route_id_enc"]  = le.transform(test["route_id"])

# ─── FEATURE ENGINEERING ───────────────────────────────────────────────────
print("Feature engineering...")
t1 = time.time()
g = train.groupby("route_id", sort=False)

train["log_target"] = np.log1p(train[TARGET].clip(lower=0))

for lag in [1, 2, 3, 4, 6, 8, 12, 24, 48, 96, 168, 336, 672]:
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for w in [4, 8, 24, 48, 96, 336]:
    train[f"lt_rm{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=1).mean())
for w in [8, 24, 96]:
    train[f"lt_rs{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=2).std())

for span in [4, 8, 24, 48]:
    train[f"lt_ewm{span}"] = g["log_target"].transform(
        lambda x: x.ewm(span=span, min_periods=1).mean())

train["lt_d1"]   = train["lt_lag1"] - train["lt_lag2"]
train["lt_d4"]   = train["lt_lag1"] - train["lt_lag4"]
train["lt_d48"]  = train["lt_lag1"] - train["lt_lag48"]
train["lt_d336"] = train["lt_lag1"] - train["lt_lag336"]
train["lt_rel8"]  = train["lt_lag1"] - train["lt_rm8"]
train["lt_rel48"] = train["lt_lag1"] - train["lt_rm48"]

for col in STATUS_COLS:
    train[f"{col}_log"] = np.log1p(train[col].clip(lower=0))
    for lag in [1, 2, 4, 8]:
        train[f"{col}_lag{lag}"] = g[col].transform("shift", lag)
    for w in [4, 8]:
        train[f"{col}_rm{w}"] = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())

train["st_local"]    = train[["status_1","status_2","status_3"]].sum(axis=1)
train["st_incoming"] = train[["status_4","status_5","status_6"]].sum(axis=1)
train["st_ratio"]    = np.log1p(train["st_incoming"]) - np.log1p(train["st_local"] + 1)

ts = train["timestamp"]
h  = ts.dt.hour + ts.dt.minute / 60
d  = ts.dt.dayofweek.astype(float)
train["hour_sin"]   = np.sin(2 * np.pi * h / 24)
train["hour_cos"]   = np.cos(2 * np.pi * h / 24)
train["dow_sin"]    = np.sin(2 * np.pi * d / 7)
train["dow_cos"]    = np.cos(2 * np.pi * d / 7)
train["is_weekend"] = (d >= 5).astype(float)

route_stats = g["log_target"].agg(["mean", "std",
                                    lambda x: x.quantile(.1),
                                    lambda x: x.quantile(.9)])
route_stats.columns = ["r_logmean", "r_logstd", "r_logq10", "r_logq90"]
route_stats = route_stats.reset_index()
train = train.merge(route_stats, on="route_id", how="left")

print(f"  Done in {time.time()-t1:.1f}s  cols={len(train.columns)}")

# ─── PER-ROUTE NORMALIZATION ───────────────────────────────────────────────
route_raw_mean = (g[TARGET].mean().clip(lower=1).rename("route_raw_mean").reset_index())
train = train.merge(route_raw_mean, on="route_id", how="left")
train["norm_log_target"] = np.log1p(
    (train[TARGET].clip(lower=0) / train["route_raw_mean"]).clip(lower=0)
)

# ─── FEATURE SET ───────────────────────────────────────────────────────────
exclude = {TARGET, "timestamp", "id", "log_target", "norm_log_target", "route_id"}
BASE_FEATS = [c for c in train.columns if c not in exclude]
print(f"  Features: {len(BASE_FEATS)}")

# ─── STEP-TIME FEATURES ────────────────────────────────────────────────────
def step_time_feats(origin_ts: pd.Series, step: int) -> pd.DataFrame:
    tgt_ts = origin_ts + pd.Timedelta(minutes=30 * step)
    h = tgt_ts.dt.hour + tgt_ts.dt.minute / 60
    d = tgt_ts.dt.dayofweek.astype(float)
    return pd.DataFrame({
        "tgt_hour_sin": np.sin(2 * np.pi * h / 24),
        "tgt_hour_cos": np.cos(2 * np.pi * h / 24),
        "tgt_dow_sin":  np.sin(2 * np.pi * d / 7),
        "tgt_dow_cos":  np.cos(2 * np.pi * d / 7),
        "tgt_is_wknd":  (d >= 5).astype(float),
        "step":         float(step),
    }, index=origin_ts.index)

# ─── VALIDATION SPLIT ──────────────────────────────────────────────────────
ts_max       = train["timestamp"].max()
CUTOFF       = ts_max - pd.Timedelta(days=7)
TRAIN_START  = CUTOFF - pd.Timedelta(days=TRAIN_DAYS)
train_mask   = (train["timestamp"] > TRAIN_START) & (train["timestamp"] <= CUTOFF)
val_mask     = train["timestamp"] >  CUTOFF
g_full       = train.groupby("route_id", sort=False)
print(f"  Training window: {TRAIN_START.date()} → {CUTOFF.date()} ({TRAIN_DAYS}d)")

# ─── BUILD TRAINING DATA ────────────────────────────────────────────────────
print("\nBuilding combined multi-step dataset...")
t2 = time.time()

all_X_tr, all_y_tr = [], []
all_X_va, all_y_va = [], []

for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)

    valid  = y_log.notna()
    mask_tr = train_mask & valid
    mask_va = val_mask   & valid

    all_X_tr.append(X_step.loc[mask_tr])
    all_y_tr.append(y_log.loc[mask_tr])
    all_X_va.append(X_step.loc[mask_va])
    all_y_va.append(y_log.loc[mask_va])

X_tr = pd.concat(all_X_tr, ignore_index=True)
y_tr = pd.concat(all_y_tr, ignore_index=True)
X_va = pd.concat(all_X_va, ignore_index=True)
y_va = pd.concat(all_y_va, ignore_index=True)

print(f"  train={len(X_tr):,}  val={len(X_va):,}  done in {time.time()-t2:.1f}s")

del all_X_tr, all_X_va, all_y_tr, all_y_va
gc.collect()

# ─── TRAIN (val model for early stopping) ──────────────────────────────────
print("\n=== Training XGBoost (Huber, GPU) ===")

model_val = XGBRegressor(
    n_estimators          = 3000,
    max_depth             = 8,
    objective             = "reg:pseudohubererror",
    huber_slope           = 1.0,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    tree_method           = "hist",
    device                = XGB_DEVICE,
    random_state          = SEED,
    early_stopping_rounds = 150,
    eval_metric           = "mae",
    verbosity             = 1,
)
model_val.fit(
    X_tr, y_tr,
    eval_set  = [(X_va, y_va)],
    verbose   = 200,
)
best_n = max(1, int(model_val.best_iteration))
print(f"\n  best_iteration = {best_n}")

pred_va_log = model_val.predict(X_va)
pred_va_raw = np.expm1(pred_va_log).clip(0)
y_va_raw    = np.expm1(y_va.values)
val_score   = wape_rbias(y_va_raw, pred_va_raw)
print(f"  Val WAPE+RBias = {val_score:.4f}")

# ─── RETRAIN (MAE, best_n iters) ───────────────────────────────────────────
print(f"\nRetraining on last {TRAIN_DAYS}d + val (iterations={best_n})...")
t3 = time.time()

full_mask = train["timestamp"] > TRAIN_START

all_X_full, all_y_full = [], []
for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)
    valid  = y_log.notna() & full_mask
    all_X_full.append(X_step.loc[valid])
    all_y_full.append(y_log.loc[valid])

X_full = pd.concat(all_X_full, ignore_index=True)
y_full = pd.concat(all_y_full, ignore_index=True)

del all_X_full, all_y_full
gc.collect()

model_final = XGBRegressor(
    n_estimators     = best_n,
    max_depth        = 8,
    objective        = "reg:absoluteerror",
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    tree_method      = "hist",
    device           = XGB_DEVICE,
    random_state     = SEED,
    verbosity        = 0,
)
model_final.fit(X_full, y_full)
print(f"  Done in {time.time()-t3:.1f}s")

model_path = "data/processed/models/xgb_v1/xgb_v1_final.ubj"
model_final.save_model(model_path)
print(f"  Saved → {model_path}")

del X_full, y_full, X_tr, y_tr, X_va, y_va
gc.collect()

# ─── PREDICT TEST ──────────────────────────────────────────────────────────
print("\n=== Predicting test ===")

last_rows_df = (train.groupby("route_id", sort=False)
                     .tail(1).reset_index(drop=True).copy())

test_routes = test["route_id"].values
test_routes_enc = test["route_id_enc"].values
_merge_cols = ["route_id"] + [f for f in BASE_FEATS if f not in ("route_id", "route_id_enc")]
test_X_base = (pd.DataFrame({"route_id": test_routes})
               .merge(last_rows_df[["route_id"] + [f for f in BASE_FEATS]], on="route_id", how="left")
               [BASE_FEATS].reset_index(drop=True))
test_X_base["route_id_enc"] = test_routes_enc

test_ts  = test["timestamp"]
step_col = ((test_ts - train["timestamp"].max()) / pd.Timedelta("30min")).astype(int)

test_preds = np.zeros(len(test))

for step in range(1, PRED_STEPS + 1):
    step_mask = (step_col == step).values
    st_df_test = step_time_feats(test_ts, step)
    X_test = pd.concat([
        test_X_base.reset_index(drop=True),
        st_df_test.reset_index(drop=True)
    ], axis=1)
    X_test = X_test[BASE_FEATS + list(st_df_test.columns)].copy()

    pred_norm = np.expm1(model_final.predict(X_test)).clip(0)
    route_means = (pd.DataFrame({"route_id": test_routes})
                   .merge(route_raw_mean, on="route_id", how="left")["route_raw_mean"].values)
    pred_raw = pred_norm * route_means
    test_preds[step_mask] = pred_raw[step_mask]

# ─── BIAS CORRECTION ──────────────────────────────────────────────────────
val_all_pred, val_all_true = [], []
for step in range(1, PRED_STEPS + 1):
    y_log  = g_full["norm_log_target"].shift(-step)
    st_df  = step_time_feats(train["timestamp"], step)
    X_step = pd.concat([train[BASE_FEATS], st_df], axis=1)
    valid  = y_log.notna() & val_mask
    X_v    = X_step.loc[valid].copy()
    p_norm = np.expm1(model_final.predict(X_v)).clip(0)
    r_means_v = train.loc[valid, "route_raw_mean"].values
    val_all_pred.append(p_norm * r_means_v)
    val_all_true.append(np.expm1(y_log.loc[valid].values) * r_means_v)

y_v_all = np.concatenate(val_all_true)
p_v_all = np.concatenate(val_all_pred)
final_val = wape_rbias(y_v_all, p_v_all)
bias      = y_v_all.sum() / max(p_v_all.sum(), 1)
print(f"  Val WAPE+RBias (final model): {final_val:.4f}")
print(f"  Bias factor: {bias:.4f}")
if abs(bias - 1) > 0.01:
    test_preds *= bias
    print(f"  Bias correction applied.")

test_preds = np.clip(test_preds, 0, None)

# ─── SUBMISSION ────────────────────────────────────────────────────────────
sub = test[["id"]].copy()
sub["y_pred"] = test_preds.round(0)
sub = sub.sort_values("id").reset_index(drop=True)

assert sub["id"].isna().sum() == 0
assert sub["y_pred"].isna().sum() == 0
assert (sub["y_pred"] < 0).sum() == 0

out = "data/processed/components/submission_xgb_v1.csv"
sub.to_csv(out, index=False)
print(f"\n{'='*60}")
print(f"Saved → {out}  shape={sub.shape}")
print(f"\nStats:\n{sub['y_pred'].describe().round(0)}")
print(f"\nTotal: {(time.time()-T0)/60:.1f} min")


In [ ]:
run_model('solution_xgb_v1.py', 'submission_xgb_v1.csv')

# MLP
Сеть 512→512→256 и embedding маршрута, 8 выходов. Доля внутри ансамбля: **20%**.

In [ ]:
%%writefile .cache/scripts/solution_mlp_v6.py
import os, time, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
T0         = time.time()
TARGET     = "target_1h"
PRED_STEPS = 8
SEED       = 42
TRAIN_DAYS = 60
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 4096
EPOCHS     = 60
LR         = 3e-4

# ── Fix all seeds ────────────────────────────────────────────────────────────
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f"Device: {DEVICE}  SEED={SEED}")

os.makedirs("data/processed/models/mlp_v6", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = y_true > 0
    wape  = np.abs(y_pred[mask] - y_true[mask]).sum() / y_true[mask].sum()
    rbias = abs(y_pred[mask].sum() / y_true[mask].sum() - 1)
    return wape + rbias

# ─── LOAD ──────────────────────────────────────────────────────────────────
print("Loading...")
train = pd.read_parquet("data/raw/train_solo_track.parquet")
test  = pd.read_parquet("data/raw/test_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"]  = pd.to_datetime(test["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test  = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"  train {train.shape}  test {test.shape}")

STATUS_COLS = [c for c in train.columns if c.startswith("status_")]

# ─── FEATURE ENGINEERING ───────────────────────────────────────────────────
print("Feature engineering...")
t1 = time.time()
g = train.groupby("route_id", sort=False)

train["log_target"] = np.log1p(train[TARGET].clip(lower=0))

for lag in [1, 2, 3, 4, 6, 8, 12, 24, 48, 96, 168, 336, 672]:
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for lag in range(40, 48):
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(328, 336):
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)
for lag in range(664, 672):
    train[f"lt_lag{lag}"] = g["log_target"].shift(lag)

for w in [4, 8, 24, 48, 96, 336]:
    train[f"lt_rm{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=1).mean())
for w in [8, 24, 96]:
    train[f"lt_rs{w}"]  = g["log_target"].transform(lambda x: x.rolling(w, min_periods=2).std())

for span in [4, 8, 24, 48]:
    train[f"lt_ewm{span}"] = g["log_target"].transform(
        lambda x: x.ewm(span=span, min_periods=1).mean())

train["lt_d1"]   = train["lt_lag1"] - train["lt_lag2"]
train["lt_d4"]   = train["lt_lag1"] - train["lt_lag4"]
train["lt_d48"]  = train["lt_lag1"] - train["lt_lag48"]
train["lt_d336"] = train["lt_lag1"] - train["lt_lag336"]
train["lt_rel8"]  = train["lt_lag1"] - train["lt_rm8"]
train["lt_rel48"] = train["lt_lag1"] - train["lt_rm48"]

for col in STATUS_COLS:
    train[f"{col}_log"] = np.log1p(train[col].clip(lower=0))
    for lag in [1, 2, 4, 8]:
        train[f"{col}_lag{lag}"] = g[col].transform("shift", lag)
    for w in [4, 8]:
        train[f"{col}_rm{w}"] = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())

train["st_local"]    = train[["status_1","status_2","status_3"]].sum(axis=1)
train["st_incoming"] = train[["status_4","status_5","status_6"]].sum(axis=1)
train["st_ratio"]    = np.log1p(train["st_incoming"]) - np.log1p(train["st_local"] + 1)

ts = train["timestamp"]
h  = ts.dt.hour + ts.dt.minute / 60
d  = ts.dt.dayofweek.astype(float)
train["hour_sin"]   = np.sin(2 * np.pi * h / 24)
train["hour_cos"]   = np.cos(2 * np.pi * h / 24)
train["dow_sin"]    = np.sin(2 * np.pi * d / 7)
train["dow_cos"]    = np.cos(2 * np.pi * d / 7)
train["is_weekend"] = (d >= 5).astype(float)

route_stats = g["log_target"].agg(["mean", "std",
                                    lambda x: x.quantile(.1),
                                    lambda x: x.quantile(.9)])
route_stats.columns = ["r_logmean", "r_logstd", "r_logq10", "r_logq90"]
route_stats = route_stats.reset_index()
train = train.merge(route_stats, on="route_id", how="left")

print(f"  Done in {time.time()-t1:.1f}s  cols={len(train.columns)}")

# ─── PER-ROUTE NORMALIZATION ───────────────────────────────────────────────
g = train.groupby("route_id", sort=False)
route_raw_mean = (g[TARGET].mean().clip(lower=1)
                  .rename("route_raw_mean").reset_index())
train = train.merge(route_raw_mean, on="route_id", how="left")
train["norm_log_target"] = np.log1p(
    (train[TARGET].clip(lower=0) / train["route_raw_mean"]).clip(lower=0)
)

# ─── ROUTE ENCODING ────────────────────────────────────────────────────────
route_ids = sorted(train["route_id"].unique())
route2idx = {r: i for i, r in enumerate(route_ids)}
N_ROUTES  = len(route_ids)
train["route_idx"] = train["route_id"].map(route2idx)

# ─── FEATURE COLUMNS ────────────────────────────────────────────────────────
exclude   = {TARGET, "timestamp", "id", "log_target", "norm_log_target",
             "route_id", "route_idx"}
FEAT_COLS = [c for c in train.columns if c not in exclude]
N_FEATS   = len(FEAT_COLS)
print(f"  Features: {N_FEATS}")

# ─── MULTI-OUTPUT TARGETS ──────────────────────────────────────────────────
print("Building multi-output targets...")
g_full = train.groupby("route_id", sort=False)
y_cols = []
for step in range(1, PRED_STEPS + 1):
    col = f"y_step{step}"
    train[col] = g_full["norm_log_target"].shift(-step)
    y_cols.append(col)

# ─── SPLITS ────────────────────────────────────────────────────────────────
ts_max      = train["timestamp"].max()
CUTOFF      = ts_max - pd.Timedelta(days=7)
TRAIN_START = CUTOFF - pd.Timedelta(days=TRAIN_DAYS)
train_mask  = (train["timestamp"] > TRAIN_START) & (train["timestamp"] <= CUTOFF)
val_mask    = train["timestamp"] > CUTOFF
y_valid     = train[y_cols].notna().all(axis=1)

idx_tr   = train.index[train_mask & y_valid]
idx_va   = train.index[val_mask   & y_valid]
idx_full = train.index[(train["timestamp"] > TRAIN_START) & y_valid]

def make_arrays(idx):
    X  = train.loc[idx, FEAT_COLS].values.astype(np.float32)
    X  = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y  = train.loc[idx, y_cols].values.astype(np.float32)
    r  = train.loc[idx, "route_idx"].values.astype(np.int64)
    rm = train.loc[idx, "route_raw_mean"].values.astype(np.float32)
    return X, y, r, rm

X_tr, y_tr, ridx_tr, rm_tr       = make_arrays(idx_tr)
X_va, y_va, ridx_va, rm_va       = make_arrays(idx_va)
X_full, y_full, ridx_full, rm_full = make_arrays(idx_full)
print(f"  train={len(X_tr):,}  val={len(X_va):,}  full={len(X_full):,}")

# ─── DATASET ───────────────────────────────────────────────────────────────
class TSDataset(Dataset):
    def __init__(self, X, y, ridx):
        self.X    = torch.from_numpy(X)
        self.y    = torch.from_numpy(y)
        self.ridx = torch.from_numpy(ridx)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.ridx[i], self.y[i]

# ─── MODEL (same as mlp_v1) ─────────────────────────────────────────────────
class MLPForecaster(nn.Module):
    def __init__(self, n_feats, n_routes, embed_dim=32, pred_steps=8):
        super().__init__()
        self.route_emb = nn.Embedding(n_routes, embed_dim)
        in_dim = n_feats + embed_dim
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Linear(in_dim, 512),  nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 512),     nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, pred_steps),
        )
    def forward(self, x, route_idx):
        emb = self.route_emb(route_idx)
        inp = torch.cat([x, emb], dim=1)
        return self.net(inp)

model = MLPForecaster(N_FEATS, N_ROUTES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Params: {n_params:,}")

loss_fn   = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# ─── TRAIN ──────────────────────────────────────────────────────────────────
print("\n=== Training MLP v6 (step-adjusted lags + per-step bias corr) ===")

def make_loader(X, y, ridx, shuffle):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(TSDataset(X, y, ridx),
                      batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0,
                      worker_init_fn=lambda wid: np.random.seed(SEED + wid),
                      generator=g if shuffle else None)

tr_loader = make_loader(X_tr, y_tr, ridx_tr, shuffle=True)
va_loader = make_loader(X_va, y_va, ridx_va, shuffle=False)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(tr_loader), epochs=EPOCHS)

best_val   = 999
best_state = None
best_epoch = 1
PATIENCE   = 8
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for xb, rb, yb in tr_loader:
        xb, rb, yb = xb.to(DEVICE), rb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb, rb)
        loss = loss_fn(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_loader)

    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xb, rb, yb in va_loader:
            p = model(xb.to(DEVICE), rb.to(DEVICE)).cpu().numpy()
            all_pred.append(p)
            all_true.append(yb.numpy())

    pred_log = np.concatenate(all_pred)    # (N, 8)
    true_log = np.concatenate(all_true)    # (N, 8)
    pred_raw = np.expm1(pred_log) * rm_va[:, None]
    true_raw = np.expm1(true_log) * rm_va[:, None]
    val_sc   = wape_rbias(true_raw.ravel(), pred_raw.ravel())

    print(f"  Epoch {epoch:2d}  loss={tr_loss:.4f}  val={val_sc:.4f}", end="")
    if val_sc < best_val:
        best_val   = val_sc
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        print("  <- best")
    else:
        no_improve += 1
        print()
        if no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nBest val = {best_val:.4f}  (epoch {best_epoch})")
model.load_state_dict(best_state)
torch.save(best_state, "data/processed/models/mlp_v6/mlp_v6_val.pt")
print("  Saved → data/processed/models/mlp_v6/mlp_v6_val.pt")

# ─── RETRAIN ON FULL ────────────────────────────────────────────────────────
print("\nRetraining on full (train+val)...")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

full_loader = make_loader(X_full, y_full, ridx_full, shuffle=True)

model2 = MLPForecaster(N_FEATS, N_ROUTES).to(DEVICE)
opt2   = torch.optim.AdamW(model2.parameters(), lr=LR, weight_decay=1e-4)

RETRAIN_EPOCHS = best_epoch + 2
sched2 = torch.optim.lr_scheduler.OneCycleLR(
    opt2, max_lr=LR,
    steps_per_epoch=len(full_loader), epochs=RETRAIN_EPOCHS)

for epoch in range(1, RETRAIN_EPOCHS + 1):
    model2.train()
    ep_loss = 0.0
    for xb, rb, yb in full_loader:
        xb, rb, yb = xb.to(DEVICE), rb.to(DEVICE), yb.to(DEVICE)
        opt2.zero_grad()
        loss = loss_fn(model2(xb, rb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        opt2.step()
        sched2.step()
        ep_loss += loss.item()
    print(f"  Retrain epoch {epoch}/{RETRAIN_EPOCHS}  loss={ep_loss/len(full_loader):.4f}")

torch.save(model2.state_dict(), "data/processed/models/mlp_v6/mlp_v6_final.pt")
print("  Saved → data/processed/models/mlp_v6/mlp_v6_final.pt")

del X_tr, y_tr, X_va, y_va, X_full, y_full
gc.collect()

# ─── PREDICT TEST ──────────────────────────────────────────────────────────
print("\nPredicting test...")
model2.eval()

last_rows   = (train.groupby("route_id", sort=False).tail(1).reset_index(drop=True))
test_routes = test["route_id"].values
all_rids    = sorted(set(test_routes))

X_test_list, ridx_test_list = [], []
for rid in all_rids:
    row = last_rows[last_rows["route_id"] == rid].iloc[0]
    x   = row[FEAT_COLS].values.astype(np.float32)
    x   = np.nan_to_num(x, nan=0.0)
    X_test_list.append(x)
    ridx_test_list.append(route2idx[rid])

X_test_arr    = np.stack(X_test_list).astype(np.float32)
ridx_test_arr = np.array(ridx_test_list, dtype=np.int64)

with torch.no_grad():
    pred_log_all = model2(
        torch.from_numpy(X_test_arr).to(DEVICE),
        torch.from_numpy(ridx_test_arr).to(DEVICE)
    ).cpu().numpy()   # (n_routes, 8)

route_raw_mean_s = route_raw_mean.set_index("route_id")["route_raw_mean"]
rm_arr   = np.array([route_raw_mean_s.loc[r] for r in all_rids])
pred_raw = np.expm1(pred_log_all) * rm_arr[:, None]  # (n_routes, 8)
route2pred = {r: pred_raw[i] for i, r in enumerate(all_rids)}

# ─── PER-STEP BIAS CORRECTION ──────────────────────────────────────────────
print("\nPer-step bias correction...")
X_va_arr, y_va_arr, ridx_va_arr, rm_va2 = make_arrays(idx_va)

with torch.no_grad():
    p_va = model2(
        torch.from_numpy(X_va_arr).to(DEVICE),
        torch.from_numpy(ridx_va_arr).to(DEVICE)
    ).cpu().numpy()   # (N_val, 8)

pred_va_raw = np.expm1(p_va) * rm_va2[:, None]   # (N_val, 8)
true_va_raw = np.expm1(y_va_arr) * rm_va2[:, None]

step_scale = []
for s in range(PRED_STEPS):
    p_s = pred_va_raw[:, s]
    t_s = true_va_raw[:, s]
    sc  = np.clip(t_s.sum() / max(p_s.sum(), 1e-6), 0.7, 1.4)
    step_scale.append(sc)
    print(f"  step {s+1}: scale={sc:.4f}  val_pred={p_s.sum():.0f}  val_true={t_s.sum():.0f}")

for s in range(PRED_STEPS):
    pred_raw[:, s] *= step_scale[s]
route2pred = {r: pred_raw[i] for i, r in enumerate(all_rids)}

overall_val = wape_rbias(true_va_raw.ravel(), pred_va_raw.ravel())
print(f"\n  Val WAPE+RBias (before bias corr): {overall_val:.4f}")

# ─── BUILD SUBMISSION ───────────────────────────────────────────────────────
test_sorted = test.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
step_col    = ((test_sorted["timestamp"] - train["timestamp"].max())
               / pd.Timedelta("30min")).astype(int).values

test_sorted["y_pred"] = [
    float(route2pred[row.route_id][step_col[i] - 1])
    for i, row in enumerate(test_sorted.itertuples())
]

sub = test_sorted[["id", "y_pred"]].copy()
sub["y_pred"] = sub["y_pred"].clip(0).round(0)
sub = sub.sort_values("id").reset_index(drop=True)

assert sub["id"].isna().sum() == 0
assert sub["y_pred"].isna().sum() == 0
assert (sub["y_pred"] < 0).sum() == 0

out = "data/processed/components/submission_mlp_v6.csv"
sub.to_csv(out, index=False)
print(f"\nSaved → {out}")
print(f"Stats:\n{sub['y_pred'].describe().round(0)}")
print(f"\nTotal: {(time.time()-T0)/60:.1f} min")


In [ ]:
run_model('solution_mlp_v6.py', 'submission_mlp_v6.csv')

# Финальный ансамбль
Внутренние веса: **30 / 30 / 20 / 10 / 10**. CatBoost88 в итоговом прогнозе: **50%** для h1, **40%** для h2 и **25%** для h3–h8.

In [ ]:
import numpy as np
import pandas as pd

test = pd.read_parquet(ROOT / "data/raw/test_solo_track.parquet")
train_last = pd.read_parquet(ROOT / "data/raw/train_solo_track.parquet", columns=["timestamp"])["timestamp"].max()
test["timestamp"] = pd.to_datetime(test["timestamp"])
horizons = ((test["timestamp"] - pd.Timestamp(train_last)) / pd.Timedelta(minutes=30)).to_numpy()
assert np.isin(horizons, range(1, 9)).all(), "Неверные горизонты в тесте"
assert len(test) == 8000 and test["id"].is_unique
assert test.groupby("route_id").size().eq(8).all(), "Требуется 8 точек на каждый маршрут"
step_by_id = test.set_index("id").assign(horizon=horizons)["horizon"].astype(int)

def read_preds(file):
    frame = pd.read_csv(COMPONENTS / file)
    assert frame.columns.tolist() == ["id", "y_pred"], file
    assert frame.id.is_unique and len(frame) == len(test), file
    aligned = frame.set_index("id").reindex(test["id"].to_numpy())
    values = aligned.y_pred.to_numpy(dtype=np.float64)
    assert np.isfinite(values).all() and (values >= 0).all(), file
    return values

p88 = read_preds("cb88.csv")
a = (0.30 * read_preds("submission_cb_v25.csv")
   + 0.30 * read_preds("submission_cb_v36.csv")
   + 0.20 * read_preds("submission_mlp_v6.csv")
   + 0.10 * read_preds("submission_lgb_v1.csv")
   + 0.10 * read_preds("submission_xgb_v1.csv"))
a = np.maximum(a, 0).round(0)  # как при исходном смешивании 5 компонентов
weights = np.array([0.50, 0.40, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25])
w = weights[horizons.astype(int) - 1]
final_pred = np.maximum(w * p88 + (1 - w) * a, 0)
assert np.isfinite(final_pred).all()
output = test[["id"]].copy()
output["y_pred"] = final_pred
output = output.sort_values("id").reset_index(drop=True)
assert output.shape == (8000, 2) and output.id.is_unique
assert output.columns.tolist() == ["id", "y_pred"]
path = ROOT / "data/processed/submission.csv"
output.to_csv(path, index=False)
print("Готово:", path)
print("Строк:", len(output), "| Сумма прогноза:", float(output.y_pred.sum()))
print("Горизонты:", dict(zip(*np.unique(horizons.astype(int), return_counts=True))))
output.head(8)
